# So sánh 100 nến: TradingView → SQL / Redis

Notebook này dùng **TradingView làm mốc tham chiếu**. Mục tiêu là trả lời nhanh:

- SQL có thiếu nến nào mà TradingView đang có không?
- Redis DB0 có thiếu nến nào mà TradingView đang có không?
- Với cùng timestamp, `open/high/low/close` trong SQL hoặc Redis lệch TradingView bao nhiêu, gồm cả `% chênh lệch`?
- Nếu SQL/Redis chỉ chậm hơn TradingView đúng 1 cây mới nhất do chu kỳ live fetching, notebook tách thành cảnh báo `allowed_one_bar_lag`, không tính là lỗi dữ liệu.

Flow chạy tuần tự: setup → lấy TradingView → lấy SQL → lấy Redis → đối chiếu. SQL và Redis là nguồn có sẵn; notebook chỉ đọc ra để so sánh, không ghi Fact/Redis. SQL batch reader của engine có dùng bảng tạm trong session để gom pair, giống cơ chế hiện có trong source.

In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path
from time import perf_counter
import sys

import pandas as pd
import plotly.graph_objects as go
import redis
from IPython.display import display
from plotly.subplots import make_subplots

sys.dont_write_bytecode = True
candidates = [Path.cwd(), Path.cwd() / 'core_program', *Path.cwd().parents]
REPOSITORY = next((path for path in candidates if (path / 'src' / 'dp_program').is_dir()), None)
if REPOSITORY is None:
    raise RuntimeError('Open this notebook from dp_program_v3, core_program, or research.')

sys.path.insert(0, str(REPOSITORY / 'src'))

from dp_program.configuration import load_config
from dp_program.engine.auth import _best_material, token_seconds_remaining
from dp_program.engine.pipeline import validate_candles
from dp_program.engine.sql_connector import read_latest_candles_for_pairs, select_pairs
from dp_program.engine.websocket import FetchRequest, _fetch_batch_once, request_key
from dp_program.util.redis_publisher import _RedisPublisher

BARS = 100
FIELDS = ('open', 'high', 'low', 'close')

config = load_config(REPOSITORY.parent / 'run_dp' / 'config.yaml')
if config['redis']['db'] != 0:
    raise RuntimeError('This notebook only reads Redis DB0.')

pairs = select_pairs(config, live=True)
pair_keys = [(int(symbol['symbol_id']), timeframe['code']) for symbol, timeframe in pairs]
pair_meta = {
    (int(symbol['symbol_id']), timeframe['code']): {
        'symbol': f"{symbol['exchange']}:{symbol['symbol']}",
        'raw_symbol': symbol['symbol'],
        'timeframe': timeframe['code'],
    }
    for symbol, timeframe in pairs
}


def elapsed(started):
    return f'{perf_counter() - started:.2f}s'


def as_time(value):
    if isinstance(value, datetime):
        return (value.astimezone(timezone.utc) if value.tzinfo else value).replace(tzinfo=None, microsecond=0)
    text = str(value or '').strip()
    if text.isdigit():
        return datetime.fromtimestamp(int(float(text)), timezone.utc).replace(tzinfo=None)
    return datetime.strptime(text, '%Y-%m-%d %H:%M:%S')


def as_ohlc(values):
    return tuple(Decimal(str(value)) for value in values)


print(f'Live pairs: {len(pairs)} | TradingView reference candles: {BARS} | Redis DB: {config["redis"]["db"]}')

In [ ]:
started = perf_counter()
_cache, auth = _best_material(config)
if auth is None or token_seconds_remaining(auth[1]) <= 300:
    raise RuntimeError('TradingView token is unavailable or expires too soon. Run normal DP authentication first.')

tv_config = dict(config['tradingview'], auth_token=auth[1], cookie=auth[2])
now = datetime.now(timezone.utc)
by_symbol = defaultdict(list)
for symbol, timeframe in pairs:
    by_symbol[(symbol['exchange'], symbol['symbol'])].append((symbol, timeframe))

tv_data = {}
tv_batches = []
for (exchange, symbol_name), group in by_symbol.items():
    batch_started = perf_counter()
    if len(group) > 15:
        raise RuntimeError('This notebook supports at most 15 timeframes per symbol.')
    requests = [FetchRequest(symbol, timeframe, BARS + 1, BARS + 1) for symbol, timeframe in group]
    fetched, metrics = _fetch_batch_once(tv_config, requests, config['backfill']['max_bars_per_request'])
    for request in requests:
        closed = validate_candles(
            fetched[request_key(request)].candles, request.timeframe, closed_only=True, now=now
        )[-BARS:]
        tv_data[request_key(request)] = {
            as_time(candle['timestamp']): as_ohlc(candle[field] for field in FIELDS)
            for candle in closed
        }
    tv_batches.append({
        'symbol': f'{exchange}:{symbol_name}',
        'series': len(requests),
        'seconds': round(perf_counter() - batch_started, 3),
        'connect_seconds': metrics.get('connect_seconds'),
        'received_bytes': metrics.get('received_bytes'),
    })

tv_batch_frame = pd.DataFrame(tv_batches)
print(f'TradingView loaded: {len(tv_data)}/{len(pairs)} pairs in {elapsed(started)}')
display(tv_batch_frame)

In [ ]:
started = perf_counter()
sql_rows = read_latest_candles_for_pairs(config, pair_keys, BARS)
sql_data = {
    key: {as_time(row[0]): as_ohlc(row[1:5]) for row in rows}
    for key, rows in sql_rows.items()
}
sql_frame = pd.DataFrame([
    {'symbol': pair_meta[key]['symbol'], 'timeframe': key[1], 'bars': len(sql_data[key])}
    for key in pair_keys
])

print(f'SQL loaded: {len(sql_data)}/{len(pairs)} pairs in {elapsed(started)}')
display(sql_frame.head(20))

In [ ]:
started = perf_counter()
settings = config['redis']
client = redis.Redis(
    host=settings['host'], port=settings['port'], db=0,
    username=settings['username'] or None, password=settings['password'] or None,
    socket_connect_timeout=5, socket_timeout=5, decode_responses=True,
)

redis_data = {key: {} for key in pair_keys}
redis_lists = []
hash_reads = []
try:
    pipe = client.pipeline(transaction=False)
    for symbol, timeframe in pairs:
        key = (int(symbol['symbol_id']), timeframe['code'])
        list_key, hash_prefix = _RedisPublisher._keys(settings, timeframe['code'], symbol['symbol'])
        redis_lists.append((key, list_key, hash_prefix))
        pipe.lrange(list_key, -BARS, -1)
    stamp_lists = pipe.execute()

    pipe = client.pipeline(transaction=False)
    for (key, _list_key, hash_prefix), stamps in zip(redis_lists, stamp_lists):
        for stamp in stamps:
            hash_reads.append((key, stamp))
            pipe.hmget(hash_prefix + stamp, ('timestamp', 'open', 'high', 'low', 'close'))
    hash_rows = pipe.execute()

    for (key, stamp), values in zip(hash_reads, hash_rows):
        timestamp, open_, high, low, close = values
        if None not in (open_, high, low, close):
            redis_data[key][as_time(timestamp or stamp)] = as_ohlc((open_, high, low, close))
finally:
    client.close()

redis_frame = pd.DataFrame([
    {'symbol': pair_meta[key]['symbol'], 'timeframe': key[1], 'bars': len(redis_data[key])}
    for key in pair_keys
])

print(f'Redis loaded: {len(redis_data)}/{len(pairs)} pairs, {len(hash_reads)} hashes in {elapsed(started)}')
display(redis_frame.head(20))

In [ ]:
def pct_from_tv(tv_value, target_value):
    if tv_value == 0:
        return None
    return float((target_value - tv_value) / abs(tv_value) * Decimal('100'))


def pct_stats(values):
    numbers = pd.to_numeric(pd.Series(list(values)), errors='coerce').dropna()
    if numbers.empty:
        return {
            'min_abs_diff_pct': None, 'max_abs_diff_pct': None,
            'avg_abs_diff_pct': None, 'median_abs_diff_pct': None,
            'p95_abs_diff_pct': None,
        }
    return {
        'min_abs_diff_pct': round(float(numbers.min()), 6),
        'max_abs_diff_pct': round(float(numbers.max()), 6),
        'avg_abs_diff_pct': round(float(numbers.mean()), 6),
        'median_abs_diff_pct': round(float(numbers.median()), 6),
        'p95_abs_diff_pct': round(float(numbers.quantile(0.95)), 6),
    }


TARGETS = (('SQL', sql_data), ('Redis', redis_data))
ALLOWED_LIVE_LAG_BARS = 1
PAIR_COLUMNS = [
    'target', 'pair', 'symbol', 'timeframe', 'status', 'tv_bars', 'target_bars',
    'allowed_lag_bars', 'missing_error_bars', 'extra_window_bars', 'ohlc_error_bars',
    'error_bars', 'error_bar_pct', 'field_diff_count',
    'min_abs_diff_pct', 'max_abs_diff_pct', 'avg_abs_diff_pct',
    'median_abs_diff_pct', 'p95_abs_diff_pct',
    'worst_bartime', 'worst_field', 'worst_tv_value', 'worst_target_value', 'worst_diff_pct',
]
DIFF_COLUMNS = [
    'pair', 'symbol', 'timeframe', 'target', 'bartime', 'issue', 'field',
    'tv_value', 'target_value', 'diff', 'diff_pct', 'abs_diff_pct',
]
REVISION_COLUMNS = ['pair', 'symbol', 'timeframe', 'bartime', 'fields', 'note']
FIELD_STATS_COLUMNS = [
    'target', 'field', 'field_diff_count',
    'min_abs_diff_pct', 'max_abs_diff_pct', 'avg_abs_diff_pct',
    'median_abs_diff_pct', 'p95_abs_diff_pct',
]
ISSUE_COLUMNS = [
    'target', 'issue', 'field_rows', 'bars',
    'min_abs_diff_pct', 'max_abs_diff_pct', 'avg_abs_diff_pct',
    'median_abs_diff_pct', 'p95_abs_diff_pct',
]
summary_rows, pair_rows, diff_rows, revision_rows = [], [], [], []

for key in pair_keys:
    meta = pair_meta[key]
    label = f"{meta['symbol']}/{meta['timeframe']}"
    tv = tv_data.get(key, {})
    sql = sql_data.get(key, {})
    redis_values = redis_data.get(key, {})
    summary_rows.append({
        'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
        'tv_reference': len(tv), 'sql': len(sql), 'redis': len(redis_values),
    })

    for target_name, target_data in TARGETS:
        target = target_data.get(key, {})
        tv_times, target_times = set(tv), set(target)
        missing_bartimes = sorted(tv_times - target_times)
        extra_bartimes = sorted(target_times - tv_times)
        field_diff_rows, ohlc_error_bars = [], 0

        for bartime in sorted(tv_times & target_times):
            bar_has_diff = False
            for index, field in enumerate(FIELDS):
                tv_value = tv[bartime][index]
                target_value = target[bartime][index]
                if tv_value == target_value:
                    continue
                bar_has_diff = True
                percent = pct_from_tv(tv_value, target_value)
                field_diff_rows.append({
                    'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'], 'target': target_name,
                    'bartime': bartime, 'issue': 'ohlc_diff', 'field': field,
                    'tv_value': tv_value, 'target_value': target_value,
                    'diff': target_value - tv_value,
                    'diff_pct': percent,
                    'abs_diff_pct': abs(percent) if percent is not None else None,
                })
            ohlc_error_bars += int(bar_has_diff)

        allowed_lag = (
            len(missing_bartimes) == ALLOWED_LIVE_LAG_BARS
            and ohlc_error_bars == 0
            and bool(tv_times)
            and missing_bartimes[-1] == max(tv_times)
            and len(extra_bartimes) <= ALLOWED_LIVE_LAG_BARS
            and all(extra < min(tv_times) for extra in extra_bartimes)
        )
        coverage_issue = 'allowed_one_bar_lag' if allowed_lag else 'missing_from_target'
        extra_issue = 'allowed_one_bar_lag' if allowed_lag else 'extra_in_target_window'

        for bartime in missing_bartimes:
            diff_rows.append({
                'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'], 'target': target_name,
                'bartime': bartime, 'issue': coverage_issue, 'field': 'timestamp',
                'tv_value': bartime, 'target_value': None, 'diff': None, 'diff_pct': None, 'abs_diff_pct': None,
            })
        for bartime in extra_bartimes:
            diff_rows.append({
                'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'], 'target': target_name,
                'bartime': bartime, 'issue': extra_issue, 'field': 'timestamp',
                'tv_value': None, 'target_value': bartime, 'diff': None, 'diff_pct': None, 'abs_diff_pct': None,
            })
        diff_rows.extend(field_diff_rows)

        allowed_lag_bars = len(missing_bartimes) if allowed_lag else 0
        missing_error_bars = 0 if allowed_lag else len(missing_bartimes)
        error_bars = missing_error_bars + ohlc_error_bars
        stat = pct_stats(row['abs_diff_pct'] for row in field_diff_rows)
        worst = max(field_diff_rows, key=lambda row: row['abs_diff_pct'] or -1, default={})
        pair_rows.append({
            'target': target_name, 'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
            'status': 'ERROR' if error_bars else ('ALLOWED_LAG' if allowed_lag_bars else 'OK'),
            'tv_bars': len(tv), 'target_bars': len(target),
            'allowed_lag_bars': allowed_lag_bars,
            'missing_error_bars': missing_error_bars,
            'extra_window_bars': len(extra_bartimes),
            'ohlc_error_bars': ohlc_error_bars,
            'error_bars': error_bars,
            'error_bar_pct': round(100 * error_bars / len(tv), 4) if tv else 0,
            'field_diff_count': len(field_diff_rows),
            **stat,
            'worst_bartime': worst.get('bartime'),
            'worst_field': worst.get('field'),
            'worst_tv_value': worst.get('tv_value'),
            'worst_target_value': worst.get('target_value'),
            'worst_diff_pct': worst.get('diff_pct'),
        })

    for bartime in sorted(set(tv) & set(sql) & set(redis_values)):
        if sql[bartime] == redis_values[bartime] and sql[bartime] != tv[bartime]:
            fields = [field for index, field in enumerate(FIELDS) if sql[bartime][index] != tv[bartime][index]]
            revision_rows.append({
                'pair': label, 'symbol': meta['symbol'], 'timeframe': meta['timeframe'],
                'bartime': bartime, 'fields': ','.join(fields),
                'note': 'SQL and Redis match each other but differ from current TradingView',
            })

summary_frame = pd.DataFrame(summary_rows)
pair_stats_frame = pd.DataFrame(pair_rows, columns=PAIR_COLUMNS)
diff_frame = pd.DataFrame(diff_rows, columns=DIFF_COLUMNS)
revision_candidates_frame = pd.DataFrame(revision_rows, columns=REVISION_COLUMNS)

overview_rows = []
for target_name, _target_data in TARGETS:
    rows = pair_stats_frame[pair_stats_frame['target'].eq(target_name)]
    field_rows = diff_frame[diff_frame['target'].eq(target_name) & diff_frame['issue'].eq('ohlc_diff')]
    tv_bars = int(rows['tv_bars'].sum())
    error_bars = int(rows['error_bars'].sum())
    overview_rows.append({
        'target': target_name,
        'pairs_checked': len(rows),
        'clean_pairs': int(rows['status'].eq('OK').sum()),
        'pairs_with_allowed_lag': int(rows['allowed_lag_bars'].gt(0).sum()),
        'pairs_with_errors': int(rows['error_bars'].gt(0).sum()),
        'tv_bars': tv_bars,
        'allowed_lag_bars': int(rows['allowed_lag_bars'].sum()),
        'missing_error_bars': int(rows['missing_error_bars'].sum()),
        'ohlc_error_bars': int(rows['ohlc_error_bars'].sum()),
        'error_bars': error_bars,
        'error_bar_pct': round(100 * error_bars / tv_bars, 4) if tv_bars else 0,
        'field_diff_count': int(rows['field_diff_count'].sum()),
        **pct_stats(field_rows['abs_diff_pct']),
    })
quality_overview_frame = pd.DataFrame(overview_rows)

issue_rows = []
if not diff_frame.empty:
    work = diff_frame.assign(_bar_key=diff_frame['pair'] + '|' + diff_frame['target'] + '|' + diff_frame['bartime'].astype(str))
    for (target_name, issue), rows in work.groupby(['target', 'issue'], sort=False):
        issue_rows.append({
            'target': target_name, 'issue': issue,
            'field_rows': len(rows), 'bars': rows['_bar_key'].nunique(),
            **pct_stats(rows['abs_diff_pct']),
        })
issue_overview_frame = pd.DataFrame(issue_rows, columns=ISSUE_COLUMNS)

field_rows = []
ohlc_rows = diff_frame[diff_frame['issue'].eq('ohlc_diff')]
if not ohlc_rows.empty:
    for (target_name, field), rows in ohlc_rows.groupby(['target', 'field'], sort=False):
        field_rows.append({'target': target_name, 'field': field, 'field_diff_count': len(rows), **pct_stats(rows['abs_diff_pct'])})
field_stats_frame = pd.DataFrame(field_rows, columns=FIELD_STATS_COLUMNS)
worst_pairs_frame = pair_stats_frame[pair_stats_frame['error_bars'].gt(0)].sort_values(
    ['error_bar_pct', 'max_abs_diff_pct'], ascending=[False, False], na_position='last'
).head(30)
allowed_lag_frame = pair_stats_frame[pair_stats_frame['allowed_lag_bars'].gt(0)]


def query_pair_stats(symbol=None, timeframe=None, target=None, status=None, limit=50):
    rows = pair_stats_frame.copy()
    if symbol:
        rows = rows[rows['symbol'].str.contains(symbol, case=False, na=False)]
    if timeframe:
        rows = rows[rows['timeframe'].eq(timeframe)]
    if target:
        rows = rows[rows['target'].eq(target)]
    if status:
        rows = rows[rows['status'].eq(status)]
    return rows.sort_values(['error_bar_pct', 'max_abs_diff_pct'], ascending=[False, False], na_position='last').head(limit)


def query_diffs(symbol=None, timeframe=None, target=None, issue=None, field=None, limit=50):
    rows = diff_frame.copy()
    if rows.empty:
        return rows
    if symbol:
        rows = rows[rows['symbol'].str.contains(symbol, case=False, na=False)]
    if timeframe:
        rows = rows[rows['timeframe'].eq(timeframe)]
    if target:
        rows = rows[rows['target'].eq(target)]
    if issue:
        rows = rows[rows['issue'].eq(issue)]
    if field:
        rows = rows[rows['field'].eq(field)]
    return rows.sort_values(['abs_diff_pct', 'bartime'], ascending=[False, False], na_position='last').head(limit)


print(f'Pairs: {len(summary_frame)} | TradingView reference candles: {BARS}')
display(quality_overview_frame)
display(worst_pairs_frame)
display(field_stats_frame)
display(issue_overview_frame)
display(allowed_lag_frame.head(50))

symbols = summary_frame['symbol'].drop_duplicates().tolist()
timeframes = summary_frame['timeframe'].drop_duplicates().tolist()
fig = make_subplots(rows=1, cols=2, subplot_titles=['TV → SQL', 'TV → Redis'], horizontal_spacing=0.06)

for position, target_name in enumerate(['SQL', 'Redis'], start=1):
    rows = pair_stats_frame[pair_stats_frame['target'] == target_name]
    error_pct_pivot = rows.pivot(index='symbol', columns='timeframe', values='error_bar_pct').reindex(index=symbols, columns=timeframes).fillna(0)
    error_bars_pivot = rows.pivot(index='symbol', columns='timeframe', values='error_bars').reindex(index=symbols, columns=timeframes).fillna(0)
    allowed_lag_pivot = rows.pivot(index='symbol', columns='timeframe', values='allowed_lag_bars').reindex(index=symbols, columns=timeframes).fillna(0)
    missing_pivot = rows.pivot(index='symbol', columns='timeframe', values='missing_error_bars').reindex(index=symbols, columns=timeframes).fillna(0)
    ohlc_error_pivot = rows.pivot(index='symbol', columns='timeframe', values='ohlc_error_bars').reindex(index=symbols, columns=timeframes).fillna(0)
    max_pct_pivot = rows.pivot(index='symbol', columns='timeframe', values='max_abs_diff_pct').reindex(index=symbols, columns=timeframes).fillna(0)
    text = [[f"E{int(error_bars_pivot.iat[row, col])}/L{int(allowed_lag_pivot.iat[row, col])}" if error_bars_pivot.iat[row, col] or allowed_lag_pivot.iat[row, col] else '' for col in range(len(timeframes))] for row in range(len(symbols))]
    customdata = [[[int(error_bars_pivot.iat[row, col]), int(allowed_lag_pivot.iat[row, col]), int(missing_pivot.iat[row, col]), int(ohlc_error_pivot.iat[row, col]), float(max_pct_pivot.iat[row, col])] for col in range(len(timeframes))] for row in range(len(symbols))]

    fig.add_trace(go.Heatmap(
        z=error_pct_pivot.to_numpy(), x=timeframes, y=symbols, text=text, texttemplate='%{text}',
        customdata=customdata, zmin=0, zmax=100,
        colorscale=[[0, '#e8f5e9'], [0.01, '#fee08b'], [1, '#d73027']],
        showscale=position == 2, colorbar={'title': '% TV bars<br>error'},
        hovertemplate='%{y}<br>%{x}<br>Error bars: %{customdata[0]} (%{z:.2f}%)<br>Allowed one-bar lag: %{customdata[1]}<br>Missing error bars: %{customdata[2]}<br>OHLC error bars: %{customdata[3]}<br>Max abs diff: %{customdata[4]:.6f}%<extra></extra>',
    ), row=1, col=position)

fig.update_layout(
    title='TradingView reference vs SQL / Redis - latest 100 closed candles',
    width=1250, height=max(480, 44 * len(symbols)),
    margin={'l': 30, 'r': 30, 't': 80, 'b': 90},
)
fig.update_xaxes(tickangle=-45)
fig.update_yaxes(autorange='reversed')
fig.show()

## Cách đọc kết quả

- `quality_overview_frame`: bảng kết luận chính theo target `SQL` và `Redis`. Bảng này cho biết tổng số pair, số pair sạch, số pair chỉ lag 1 bar được phép, số pair lỗi, tổng bar lỗi, tỷ trọng lỗi, và thống kê `%` lệch OHLC.
- `worst_pairs_frame`: các symbol/timeframe lỗi nặng nhất, sắp theo `error_bar_pct` rồi `max_abs_diff_pct`.
- `field_stats_frame`: thống kê mức lệch theo từng field `open/high/low/close`, gồm min/max/trung bình/trung vị/p95 của `abs_diff_pct`.
- `issue_overview_frame`: gom số lượng theo loại issue: `ohlc_diff`, `missing_from_target`, `allowed_one_bar_lag`, `extra_in_target_window`.
- `allowed_lag_frame`: các pair đang ở trạng thái lag 1 bar được phép (trích từ `pair_stats_frame`), dùng để xem nhanh pair nào đang chờ cây mới nhất mà không cần lọc thủ công.
- Nhãn heatmap `E/L`: `E` là số bar lỗi thật, `L` là số bar lag 1 bar được phép.
- `allowed_one_bar_lag`: SQL/Redis chỉ thiếu đúng cây mới nhất của TradingView, không có lệch OHLC ở phần timestamp chung, và phần extra nếu có chỉ là cây cũ bị trượt khỏi cửa sổ 100 nến TV. Trường hợp này là cảnh báo vận hành, không tính vào `% TV bars affected`.
- `missing_error_bars`: TradingView có timestamp đó, nhưng SQL/Redis không có trong 100 nến đọc ra và không thuộc trường hợp lag 1 bar được phép.
- `ohlc_error_bars`: cùng timestamp với TradingView nhưng khác ít nhất một giá OHLC.
- `error_bar_pct`: `error_bars / tv_bars * 100`, tức tỷ trọng bar lỗi so với 100 bar TradingView làm chuẩn.
- `extra_in_target_window`: SQL/Redis có timestamp không nằm trong 100 nến TradingView hiện tại; đây là tín hiệu phụ, không tính vào `% TV bars affected`.
- `diff_frame`: bảng chi tiết từng bar/từng field. `diff = target_value - tv_value`; `diff_pct` là phần trăm lệch có dấu so với TradingView; `abs_diff_pct` dùng cho thống kê mức độ lệch.
- `revision_candidates_frame`: các bar mà SQL và Redis khớp nhau nhưng cùng khác TradingView hiện tại. Đây là nhóm nên xem kỹ để đánh giá khả năng TradingView đã chỉnh dữ liệu lịch sử sau khi DP từng lấy xuống.
- Dùng `query_pair_stats(symbol='BTCUSD', target='SQL')` để xem thống kê theo pair; dùng `query_diffs(symbol='BTCUSD', timeframe='H1', target='SQL', issue='ohlc_diff', limit=100)` để xem chi tiết bar lệch.